In [ ]:
# | default_exp preprocessing.ocr_unlimited

In [ ]:
%load_ext autoreload
%autoreload 2

# Baidu Unlimited-OCR document parsing

> Recursively parse PDF pages and images into Markdown with a remote
> [Baidu Unlimited-OCR](https://github.com/baidu/Unlimited-OCR) vLLM service.

The default service is `172.27.74.16:7870`. The notebook uses the model's
required vLLM recipe: a literal `<image>` prompt prefix,
`skip_special_tokens=False`, and the per-request no-repeat n-gram parameters.
Each PDF page is sent as an independent single-image request, so pages can run
concurrently while the resulting Markdown remains in source order.

Each input produces a UTF-8 Markdown file below `.md_unlimited/` and a sibling
`.unlimited.json` provenance file. The sidecar retains the model's raw grounded
output; Markdown receives a cleaned form with `<|ref|>` wrappers unwrapped and
`<|det|>` coordinate boxes removed. Existing outputs are skipped unless
`overwrite=True`, and completed output pairs are published atomically.

The service currently requires authentication. Put
`UNLIMITED_OCR_API_KEY=...` in the project-root `.env`, or pass `api_key=...`.
`UNLIMITED_OCR_BASE_URL` and `UNLIMITED_OCR_MODEL` can override the endpoint
and served model name. API keys are never written to output metadata.

In [ ]:
# | export
import asyncio
import base64
import json
import os
import re
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from tempfile import TemporaryDirectory
from time import perf_counter
from typing import Any, Callable, Literal
from urllib.parse import urlsplit

import httpx
import pymupdf
from dotenv import load_dotenv
from openai import AsyncOpenAI
from PIL import Image
from tqdm.auto import tqdm


In [ ]:
# | export
def _find_project_root() -> Path:
    """Find the nearest parent containing pyproject.toml."""
    starts: list[Path] = []
    module_file = globals().get("__file__")
    if isinstance(module_file, str):
        starts.append(Path(module_file).resolve().parent)
    starts.append(Path.cwd().resolve())
    for start in starts:
        for candidate in [start, *start.parents]:
            if (candidate / "pyproject.toml").is_file():
                return candidate
    return Path.cwd().resolve()


PROJ_ROOT = _find_project_root()
load_dotenv(PROJ_ROOT / ".env", override=False)

In [ ]:
# | export
@dataclass(frozen=True)
class UnlimitedOCRResult:
    """Outcome of parsing one PDF or image with Unlimited-OCR."""

    source_path: Path
    markdown_path: Path
    metadata_path: Path
    status: Literal["processed", "skipped", "failed"]
    pages_total: int = 0
    pages_completed: int = 0
    elapsed_s: float = 0.0
    error: str | None = None


_SUPPORTED_SUFFIXES = frozenset(
    {".pdf", ".png", ".jpg", ".jpeg", ".webp", ".bmp"}
)
_IMAGE_SUFFIXES = _SUPPORTED_SUFFIXES - {".pdf"}
_DEFAULT_BASE_URL = "172.27.74.16:7870"
_DEFAULT_MODEL = "baidu/Unlimited-OCR"
_DEFAULT_PROMPT = "<image>document parsing."
_DEFAULT_MAX_TOKENS = 8192
_DEFAULT_NGRAM_SIZE = 35
_DEFAULT_NGRAM_WINDOW = 128
_DEFAULT_MAX_DATA_URL_BYTES = 64 * 1024 * 1024

In [ ]:
# | export
def _resolve_root(root_folder: Path | str) -> Path:
    root = Path(root_folder).expanduser().resolve()
    if not root.exists():
        raise FileNotFoundError(f"OCR root does not exist: {root}")
    if not root.is_dir():
        raise NotADirectoryError(f"OCR root is not a directory: {root}")
    return root


def _resolve_output_dir_name(output_dir_name: str) -> str:
    normalized = output_dir_name.strip()
    path = Path(normalized)
    if (
        not normalized
        or path.is_absolute()
        or len(path.parts) != 1
        or normalized in {".", ".."}
    ):
        raise ValueError("output_dir_name must be one relative directory name")
    return normalized


def _metadata_path(markdown_path: Path) -> Path:
    return markdown_path.with_suffix(".unlimited.json")


def _ocr_jobs(
    root: Path,
    output_dir_name: str,
) -> list[tuple[Path, Path, Path]]:
    """Return deterministic source/Markdown/metadata triples."""
    output_root = root / _resolve_output_dir_name(output_dir_name)
    sources = [
        path
        for path in root.rglob("*")
        if path.is_file()
        and path.suffix.casefold() in _SUPPORTED_SUFFIXES
        and not path.is_relative_to(output_root)
    ]
    sources.sort(
        key=lambda path: (
            path.relative_to(root).as_posix().casefold(),
            path.relative_to(root).as_posix(),
        )
    )

    jobs: list[tuple[Path, Path, Path]] = []
    targets: dict[str, Path] = {}
    for source_path in sources:
        relative = source_path.relative_to(root).with_suffix(".md")
        markdown_path = output_root / relative
        collision_key = markdown_path.as_posix().casefold()
        if previous := targets.get(collision_key):
            raise ValueError(
                f"OCR output collision: {previous} and {source_path} both map to "
                f"{markdown_path}"
            )
        targets[collision_key] = source_path
        jobs.append((source_path, markdown_path, _metadata_path(markdown_path)))
    return jobs

In [ ]:
# | export
def _resolve_base_url(base_url: str | None) -> str:
    resolved = (
        base_url.strip()
        if base_url is not None
        else os.getenv("UNLIMITED_OCR_BASE_URL", "").strip() or _DEFAULT_BASE_URL
    )
    if not resolved:
        raise ValueError("Unlimited-OCR base URL must not be empty")
    if "://" not in resolved:
        resolved = f"http://{resolved}"
    resolved = resolved.rstrip("/")
    parsed = urlsplit(resolved)
    if (
        parsed.scheme not in {"http", "https"}
        or not parsed.hostname
        or parsed.username is not None
        or parsed.password is not None
        or parsed.query
        or parsed.fragment
    ):
        raise ValueError(
            "Unlimited-OCR base URL must be an HTTP(S) endpoint without "
            "credentials, query, or fragment"
        )
    if not parsed.path:
        resolved += "/v1"
    elif not parsed.path.rstrip("/").endswith("/v1"):
        resolved += "/v1"
    return resolved


def _resolve_api_key(api_key: str | None) -> str:
    if api_key is not None:
        if not api_key.strip():
            raise ValueError("api_key must not be empty")
        return api_key.strip()
    return os.getenv("UNLIMITED_OCR_API_KEY", "").strip() or "EMPTY"


def _resolve_model(model: str | None) -> str:
    if model is not None:
        if not model.strip():
            raise ValueError("model must not be empty")
        return model.strip()
    return os.getenv("UNLIMITED_OCR_MODEL", "").strip() or _DEFAULT_MODEL


def _resolve_prompt(prompt: str) -> str:
    normalized = prompt.strip()
    if not normalized.startswith("<image>"):
        raise ValueError("Unlimited-OCR prompt must begin with the literal <image>")
    return normalized


def _positive_int(value: int | None, env_name: str, default: int) -> int:
    if value is None:
        configured = os.getenv(env_name, "").strip()
        if configured:
            try:
                value = int(configured)
            except ValueError as error:
                raise ValueError(f"{env_name} must be a positive integer") from error
        else:
            value = default
    if value <= 0:
        raise ValueError(f"{env_name} must be a positive integer")
    return value


def _create_client(
    base_url: str | None,
    api_key: str | None,
    request_timeout_s: float,
) -> AsyncOpenAI:
    """Create a proxy-free client suitable for a LAN vLLM endpoint."""
    resolved_base_url = _resolve_base_url(base_url)
    http_client = httpx.AsyncClient(
        timeout=request_timeout_s,
        trust_env=False,
    )
    return AsyncOpenAI(
        api_key=_resolve_api_key(api_key),
        base_url=resolved_base_url,
        timeout=request_timeout_s,
        max_retries=0,
        http_client=http_client,
    )


async def list_served_models(
    *,
    base_url: str | None = None,
    api_key: str | None = None,
    request_timeout_s: float = 30,
) -> list[str]:
    """Return model IDs advertised by the remote OpenAI-compatible service."""
    if request_timeout_s <= 0:
        raise ValueError("request_timeout_s must be greater than zero")
    client = _create_client(base_url, api_key, request_timeout_s)
    try:
        response = await client.models.list()
        return [model.id for model in response.data]
    finally:
        await client.close()

In [ ]:
# | export
def _load_image_pixmap(path: Path) -> pymupdf.Pixmap:
    """Load a supported image at native resolution on an opaque white canvas."""
    try:
        with Image.open(path) as image:
            image.load()
            rgba = image.convert("RGBA")
            background = Image.new("RGBA", rgba.size, (255, 255, 255, 255))
            rgb = Image.alpha_composite(background, rgba).convert("RGB")
            return pymupdf.Pixmap(
                pymupdf.csRGB,
                rgb.width,
                rgb.height,
                rgb.tobytes(),
                False,
            )
    except Exception as error:
        raise ValueError(f"Could not decode image {path}: {error}") from error


def _data_url(image_bytes: bytes, mime_type: str) -> str:
    encoded = base64.b64encode(image_bytes).decode("ascii")
    return f"data:{mime_type};base64,{encoded}"


def _image_data_url(
    pixmap: pymupdf.Pixmap,
    max_data_url_bytes: int = _DEFAULT_MAX_DATA_URL_BYTES,
) -> tuple[str, str]:
    if max_data_url_bytes <= 0:
        raise ValueError("max_data_url_bytes must be greater than zero")
    png_url = _data_url(pixmap.tobytes("png"), "image/png")
    if len(png_url) <= max_data_url_bytes:
        return png_url, "image/png"

    jpeg_url = _data_url(
        pixmap.tobytes("jpeg", jpg_quality=92),
        "image/jpeg",
    )
    if len(jpeg_url) <= max_data_url_bytes:
        return jpeg_url, "image/jpeg"
    raise ValueError(
        "Image payload exceeds max_data_url_bytes after JPEG fallback; "
        "reduce PDF dpi or the source image dimensions"
    )

In [ ]:
# | export
_REF_RE = re.compile(r"<\|ref\|>(.*?)<\|/ref\|>", re.DOTALL)
_DET_RE = re.compile(r"<\|det\|>.*?<\|/det\|>", re.DOTALL)


def clean_grounding(raw: str) -> str:
    """Unwrap reference text and remove Unlimited-OCR coordinate markers."""
    cleaned = _REF_RE.sub(lambda match: match.group(1), raw)
    cleaned = _DET_RE.sub("", cleaned)
    lines = [line.rstrip() for line in cleaned.replace("\r\n", "\n").split("\n")]
    normalized: list[str] = []
    blank = False
    for line in lines:
        if line:
            normalized.append(line)
            blank = False
        elif normalized and not blank:
            normalized.append("")
            blank = True
    return "\n".join(normalized).strip()


def _value(obj: object, name: str, default: Any = None) -> Any:
    if isinstance(obj, dict):
        return obj.get(name, default)
    return getattr(obj, name, default)


def _response_record(response: object) -> tuple[str, dict[str, Any]]:
    choices = _value(response, "choices")
    if not isinstance(choices, (list, tuple)) or not choices:
        raise ValueError("Unlimited-OCR returned no choices")
    choice = choices[0]
    message = _value(choice, "message")
    content = _value(message, "content")
    if not isinstance(content, str) or not content.strip():
        raise ValueError("Unlimited-OCR returned an empty response")

    usage_obj = _value(response, "usage")
    usage: dict[str, int] = {}
    if usage_obj is not None:
        for name in ("prompt_tokens", "completion_tokens", "total_tokens"):
            count = _value(usage_obj, name)
            if isinstance(count, int):
                usage[name] = count
    return content, {
        "response_id": _value(response, "id"),
        "finish_reason": _value(choice, "finish_reason"),
        "usage": usage,
    }


def _vllm_extra_body(ngram_size: int, ngram_window: int) -> dict[str, Any]:
    if ngram_size <= 0:
        raise ValueError("ngram_size must be greater than zero")
    if ngram_window <= 0:
        raise ValueError("ngram_window must be greater than zero")
    return {
        "skip_special_tokens": False,
        "vllm_xargs": {
            "ngram_size": ngram_size,
            "window_size": ngram_window,
        },
    }


async def _parse_pixmap(
    pixmap: pymupdf.Pixmap,
    page_number: int,
    *,
    client: AsyncOpenAI,
    model: str,
    prompt: str,
    max_tokens: int,
    ngram_size: int,
    ngram_window: int,
    max_data_url_bytes: int,
    clean_output: bool,
    request_semaphore: asyncio.Semaphore | None,
) -> tuple[str, dict[str, Any]]:
    data_url, mime_type = _image_data_url(pixmap, max_data_url_bytes)
    started_at = perf_counter()

    async def send_request() -> object:
        return await client.chat.completions.create(
            model=model,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": prompt},
                        {"type": "image_url", "image_url": {"url": data_url}},
                    ],
                }
            ],
            max_tokens=max_tokens,
            temperature=0.0,
            extra_body=_vllm_extra_body(ngram_size, ngram_window),
        )

    if request_semaphore is None:
        response = await send_request()
    else:
        async with request_semaphore:
            response = await send_request()
    raw_content, response_metadata = _response_record(response)
    content = clean_grounding(raw_content) if clean_output else raw_content.strip()
    if not content:
        raise ValueError("Unlimited-OCR output is empty after post-processing")
    return content, {
        "page_number": page_number,
        "width": pixmap.width,
        "height": pixmap.height,
        "input_mime_type": mime_type,
        "elapsed_s": perf_counter() - started_at,
        "raw_content": raw_content,
        "content": content,
        **response_metadata,
    }

In [ ]:
# | export
def _source_signature(source: Path) -> dict[str, Any]:
    stat = source.stat()
    return {
        "path": str(source),
        "size": stat.st_size,
        "mtime_ns": stat.st_mtime_ns,
    }


def _publish_output_pair(
    markdown_path: Path,
    markdown: str,
    metadata: dict[str, Any],
) -> None:
    """Publish metadata first and Markdown last, restoring old outputs on error."""
    metadata_path = _metadata_path(markdown_path)
    markdown_path.parent.mkdir(parents=True, exist_ok=True)
    with TemporaryDirectory(
        dir=markdown_path.parent,
        prefix=f".{markdown_path.stem}.unlimited-",
    ) as temporary_directory:
        stage_root = Path(temporary_directory)
        stage_markdown = stage_root / markdown_path.name
        stage_metadata = stage_root / metadata_path.name
        stage_markdown.write_text(markdown, encoding="utf-8", newline="\n")
        stage_metadata.write_text(
            json.dumps(metadata, ensure_ascii=False, indent=2) + "\n",
            encoding="utf-8",
            newline="\n",
        )

        publications = [
            (stage_metadata, metadata_path),
            (stage_markdown, markdown_path),
        ]
        backups: list[tuple[Path, Path]] = []
        published: list[tuple[Path, Path]] = []
        try:
            for index, (_, final_path) in enumerate(publications):
                if final_path.exists():
                    backup_path = stage_root / f"backup-{index}"
                    final_path.replace(backup_path)
                    backups.append((backup_path, final_path))
            for staged_path, final_path in publications:
                staged_path.replace(final_path)
                published.append((final_path, staged_path))
        except Exception:
            for final_path, staged_path in reversed(published):
                if final_path.exists():
                    final_path.replace(staged_path)
            for backup_path, final_path in reversed(backups):
                if backup_path.exists():
                    backup_path.replace(final_path)
            raise


def _target_state(
    source: Path,
    target: Path,
    *,
    overwrite: bool,
) -> UnlimitedOCRResult | None:
    metadata_path = _metadata_path(target)
    if target.exists() and not target.is_file():
        return UnlimitedOCRResult(
            source,
            target,
            metadata_path,
            "failed",
            error="Markdown target is not a file",
        )
    if metadata_path.exists() and not metadata_path.is_file():
        return UnlimitedOCRResult(
            source,
            target,
            metadata_path,
            "failed",
            error="Metadata target is not a file",
        )
    if target.is_file() and not overwrite:
        return UnlimitedOCRResult(source, target, metadata_path, "skipped")
    return None


def _assembled_markdown(page_contents: list[str]) -> str:
    sections = [
        f"<!-- Page {page_number} -->\n\n{content}"
        for page_number, content in enumerate(page_contents, start=1)
    ]
    return "\n\n".join(sections).rstrip() + "\n"


def _document_metadata(
    source: Path,
    *,
    service_base_url: str,
    model: str,
    prompt: str,
    dpi: int | None,
    max_tokens: int,
    ngram_size: int,
    ngram_window: int,
    clean_output: bool,
    elapsed_s: float,
    pages: list[dict[str, Any]],
) -> dict[str, Any]:
    return {
        "schema_version": 1,
        "provider": "baidu/Unlimited-OCR",
        "source": str(source),
        "source_signature": _source_signature(source),
        "service_base_url": service_base_url,
        "model": model,
        "prompt": prompt,
        "dpi": dpi,
        "max_tokens": max_tokens,
        "decode_recipe": {
            "skip_special_tokens": False,
            "ngram_size": ngram_size,
            "ngram_window": ngram_window,
        },
        "clean_output": clean_output,
        "elapsed_s": elapsed_s,
        "pages": pages,
    }

In [ ]:
# | export
async def ocr_pdf(
    pdf_path: Path | str,
    markdown_path: Path | str,
    *,
    client: AsyncOpenAI,
    service_base_url: str = _DEFAULT_BASE_URL,
    model: str | None = None,
    prompt: str = _DEFAULT_PROMPT,
    dpi: int = 300,
    overwrite: bool = False,
    page_concurrency: int = 2,
    max_tokens: int = _DEFAULT_MAX_TOKENS,
    ngram_size: int = _DEFAULT_NGRAM_SIZE,
    ngram_window: int = _DEFAULT_NGRAM_WINDOW,
    max_data_url_bytes: int = _DEFAULT_MAX_DATA_URL_BYTES,
    clean_output: bool = True,
    request_semaphore: asyncio.Semaphore | None = None,
    page_started: Callable[[int], None] | None = None,
    page_progress: Callable[[int, int, float], None] | None = None,
) -> UnlimitedOCRResult:
    """Parse one PDF into page-ordered Unlimited-OCR Markdown."""
    source = Path(pdf_path).expanduser().resolve()
    target = Path(markdown_path).expanduser().resolve()
    metadata_path = _metadata_path(target)
    selected_base_url = _resolve_base_url(service_base_url)
    selected_model = _resolve_model(model)
    selected_prompt = _resolve_prompt(prompt)
    if dpi <= 0:
        raise ValueError("dpi must be greater than zero")
    if page_concurrency <= 0:
        raise ValueError("page_concurrency must be greater than zero")
    if max_tokens <= 0:
        raise ValueError("max_tokens must be greater than zero")
    _vllm_extra_body(ngram_size, ngram_window)
    if state := _target_state(source, target, overwrite=overwrite):
        return state

    started_at = perf_counter()
    pages_total = 0
    pages_completed = 0
    tasks: list[asyncio.Task[tuple[int, str, dict[str, Any]]]] = []
    try:
        if not source.is_file():
            raise FileNotFoundError(f"PDF does not exist: {source}")
        page_gate = asyncio.Semaphore(page_concurrency)
        with pymupdf.open(source) as document:
            if document.needs_pass:
                raise ValueError("PDF requires a password")
            pages_total = document.page_count
            if pages_total == 0:
                raise ValueError("PDF contains no pages")
            if page_started is not None:
                page_started(pages_total)

            async def process_page(
                page_number: int,
            ) -> tuple[int, str, dict[str, Any]]:
                nonlocal pages_completed
                page_started_at = perf_counter()
                async with page_gate:
                    page = document.load_page(page_number - 1)
                    pixmap = page.get_pixmap(dpi=dpi, alpha=False)
                    content, record = await _parse_pixmap(
                        pixmap,
                        page_number,
                        client=client,
                        model=selected_model,
                        prompt=selected_prompt,
                        max_tokens=max_tokens,
                        ngram_size=ngram_size,
                        ngram_window=ngram_window,
                        max_data_url_bytes=max_data_url_bytes,
                        clean_output=clean_output,
                        request_semaphore=request_semaphore,
                    )
                elapsed_s = perf_counter() - page_started_at
                record["elapsed_s"] = elapsed_s
                pages_completed += 1
                if page_progress is not None:
                    page_progress(pages_completed, pages_total, elapsed_s)
                return page_number, content, record

            tasks = [
                asyncio.create_task(process_page(page_number))
                for page_number in range(1, pages_total + 1)
            ]
            try:
                parsed_pages = await asyncio.gather(*tasks)
            finally:
                for task in tasks:
                    if not task.done():
                        task.cancel()
                await asyncio.gather(*tasks, return_exceptions=True)

        parsed_pages.sort(key=lambda item: item[0])
        document_elapsed_s = perf_counter() - started_at
        _publish_output_pair(
            target,
            _assembled_markdown([content for _, content, _ in parsed_pages]),
            _document_metadata(
                source,
                service_base_url=selected_base_url,
                model=selected_model,
                prompt=selected_prompt,
                dpi=dpi,
                max_tokens=max_tokens,
                ngram_size=ngram_size,
                ngram_window=ngram_window,
                clean_output=clean_output,
                elapsed_s=document_elapsed_s,
                pages=[record for _, _, record in parsed_pages],
            ),
        )
        return UnlimitedOCRResult(
            source,
            target,
            metadata_path,
            "processed",
            pages_total,
            pages_completed,
            perf_counter() - started_at,
        )
    except Exception as error:
        return UnlimitedOCRResult(
            source,
            target,
            metadata_path,
            "failed",
            pages_total,
            pages_completed,
            perf_counter() - started_at,
            f"{type(error).__name__}: {error}",
        )

In [ ]:
# | export
async def ocr_image(
    image_path: Path | str,
    markdown_path: Path | str,
    *,
    client: AsyncOpenAI,
    service_base_url: str = _DEFAULT_BASE_URL,
    model: str | None = None,
    prompt: str = _DEFAULT_PROMPT,
    overwrite: bool = False,
    max_tokens: int = _DEFAULT_MAX_TOKENS,
    ngram_size: int = _DEFAULT_NGRAM_SIZE,
    ngram_window: int = _DEFAULT_NGRAM_WINDOW,
    max_data_url_bytes: int = _DEFAULT_MAX_DATA_URL_BYTES,
    clean_output: bool = True,
    request_semaphore: asyncio.Semaphore | None = None,
) -> UnlimitedOCRResult:
    """Parse one image into Unlimited-OCR Markdown at native resolution."""
    source = Path(image_path).expanduser().resolve()
    target = Path(markdown_path).expanduser().resolve()
    metadata_path = _metadata_path(target)
    selected_base_url = _resolve_base_url(service_base_url)
    selected_model = _resolve_model(model)
    selected_prompt = _resolve_prompt(prompt)
    if max_tokens <= 0:
        raise ValueError("max_tokens must be greater than zero")
    _vllm_extra_body(ngram_size, ngram_window)
    if state := _target_state(source, target, overwrite=overwrite):
        return state

    started_at = perf_counter()
    pages_total = 0
    pages_completed = 0
    try:
        if not source.is_file():
            raise FileNotFoundError(f"Image does not exist: {source}")
        if source.suffix.casefold() not in _IMAGE_SUFFIXES:
            raise ValueError(f"Unsupported image type: {source.suffix or '<none>'}")
        pixmap = _load_image_pixmap(source)
        pages_total = 1
        content, page_record = await _parse_pixmap(
            pixmap,
            1,
            client=client,
            model=selected_model,
            prompt=selected_prompt,
            max_tokens=max_tokens,
            ngram_size=ngram_size,
            ngram_window=ngram_window,
            max_data_url_bytes=max_data_url_bytes,
            clean_output=clean_output,
            request_semaphore=request_semaphore,
        )
        pages_completed = 1
        document_elapsed_s = perf_counter() - started_at
        _publish_output_pair(
            target,
            _assembled_markdown([content]),
            _document_metadata(
                source,
                service_base_url=selected_base_url,
                model=selected_model,
                prompt=selected_prompt,
                dpi=None,
                max_tokens=max_tokens,
                ngram_size=ngram_size,
                ngram_window=ngram_window,
                clean_output=clean_output,
                elapsed_s=document_elapsed_s,
                pages=[page_record],
            ),
        )
        return UnlimitedOCRResult(
            source,
            target,
            metadata_path,
            "processed",
            pages_total,
            pages_completed,
            perf_counter() - started_at,
        )
    except Exception as error:
        return UnlimitedOCRResult(
            source,
            target,
            metadata_path,
            "failed",
            pages_total,
            pages_completed,
            perf_counter() - started_at,
            f"{type(error).__name__}: {error}",
        )

In [ ]:
# | export
async def ocr_folder(
    root_folder: Path | str,
    *,
    base_url: str | None = None,
    api_key: str | None = None,
    model: str | None = None,
    output_dir_name: str = ".md_unlimited",
    prompt: str = _DEFAULT_PROMPT,
    dpi: int = 300,
    overwrite: bool = False,
    request_timeout_s: float = 1200,
    max_concurrency: int | None = None,
    page_concurrency: int | None = None,
    max_tokens: int = _DEFAULT_MAX_TOKENS,
    ngram_size: int = _DEFAULT_NGRAM_SIZE,
    ngram_window: int = _DEFAULT_NGRAM_WINDOW,
    max_data_url_bytes: int = _DEFAULT_MAX_DATA_URL_BYTES,
    clean_output: bool = True,
    show_progress: bool = True,
    client: AsyncOpenAI | None = None,
) -> list[UnlimitedOCRResult]:
    """Concurrently parse PDFs and images below a folder with Unlimited-OCR."""
    root = _resolve_root(root_folder)
    selected_output_dir = _resolve_output_dir_name(output_dir_name)
    selected_base_url = _resolve_base_url(base_url)
    selected_model = _resolve_model(model)
    selected_prompt = _resolve_prompt(prompt)
    if dpi <= 0:
        raise ValueError("dpi must be greater than zero")
    if request_timeout_s <= 0:
        raise ValueError("request_timeout_s must be greater than zero")
    if max_tokens <= 0:
        raise ValueError("max_tokens must be greater than zero")
    _vllm_extra_body(ngram_size, ngram_window)
    selected_max_concurrency = _positive_int(
        max_concurrency,
        "UNLIMITED_OCR_MAX_CONCURRENCY",
        4,
    )
    selected_page_concurrency = _positive_int(
        page_concurrency,
        "UNLIMITED_OCR_PAGE_CONCURRENCY",
        2,
    )

    jobs = _ocr_jobs(root, selected_output_dir)
    if not jobs:
        print(f"No supported PDF or image files found under {root}")
        return []

    results_by_index: dict[int, UnlimitedOCRResult] = {}
    pending_jobs: list[tuple[int, Path, Path, Path]] = []
    for index, (source_path, markdown_path, metadata_path) in enumerate(jobs):
        if not overwrite and markdown_path.is_file():
            results_by_index[index] = UnlimitedOCRResult(
                source_path,
                markdown_path,
                metadata_path,
                "skipped",
            )
        else:
            pending_jobs.append((index, source_path, markdown_path, metadata_path))

    if not pending_jobs:
        results = [results_by_index[index] for index in range(len(jobs))]
        print(
            f"Unlimited-OCR complete: 0 processed, {len(results)} skipped, 0 failed"
        )
        return results

    parser_client = (
        client
        if client is not None
        else _create_client(selected_base_url, api_key, request_timeout_s)
    )
    file_semaphore = asyncio.Semaphore(selected_max_concurrency)
    request_semaphore = asyncio.Semaphore(selected_max_concurrency)
    progress = tqdm(
        total=len(pending_jobs),
        desc=f"Unlimited-OCR files ({selected_base_url})",
        unit="file",
        disable=not show_progress,
        dynamic_ncols=True,
    )

    async def run_job(
        index: int,
        source_path: Path,
        markdown_path: Path,
        metadata_path: Path,
    ) -> tuple[int, UnlimitedOCRResult]:
        async with file_semaphore:
            try:
                common = {
                    "client": parser_client,
                    "service_base_url": selected_base_url,
                    "model": selected_model,
                    "prompt": selected_prompt,
                    "overwrite": overwrite,
                    "max_tokens": max_tokens,
                    "ngram_size": ngram_size,
                    "ngram_window": ngram_window,
                    "max_data_url_bytes": max_data_url_bytes,
                    "clean_output": clean_output,
                    "request_semaphore": request_semaphore,
                }
                if source_path.suffix.casefold() == ".pdf":
                    result = await ocr_pdf(
                        source_path,
                        markdown_path,
                        dpi=dpi,
                        page_concurrency=selected_page_concurrency,
                        **common,
                    )
                else:
                    result = await ocr_image(
                        source_path,
                        markdown_path,
                        **common,
                    )
            except Exception as error:
                result = UnlimitedOCRResult(
                    source_path,
                    markdown_path,
                    metadata_path,
                    "failed",
                    error=f"{type(error).__name__}: {error}",
                )
            finally:
                progress.update(1)
            return index, result

    tasks = [
        asyncio.create_task(run_job(index, source, markdown, metadata))
        for index, source, markdown, metadata in pending_jobs
    ]
    completion_records: list[str] = []
    try:
        for completed_task in asyncio.as_completed(tasks):
            index, result = await completed_task
            results_by_index[index] = result
            error_text = f" | error={result.error}" if result.error else ""
            completion_records.append(
                f"Unlimited-OCR task: {result.source_path} | "
                f"status={result.status} | elapsed={result.elapsed_s:.2f}s"
                f"{error_text}"
            )
    finally:
        for task in tasks:
            if not task.done():
                task.cancel()
        await asyncio.gather(*tasks, return_exceptions=True)
        progress.close()
        if client is None:
            await parser_client.close()

    for record in completion_records:
        print(record)
    results = [results_by_index[index] for index in range(len(jobs))]
    counts = Counter(result.status for result in results)
    print(
        "Unlimited-OCR complete: "
        f"{counts['processed']} processed, "
        f"{counts['skipped']} skipped, "
        f"{counts['failed']} failed"
    )
    return results

## Configuration and batch execution

The health endpoint at the default host does not require authentication, but
its `/v1` API does. Add the bearer token to the project `.env`:

```dotenv
UNLIMITED_OCR_API_KEY=replace-with-the-service-token
# Optional when the deployment uses a served-model alias:
# UNLIMITED_OCR_MODEL=Unlimited-OCR
```

Run `list_served_models()` first to confirm the model ID. The real OCR calls
below are opt-in because they can be long-running and write generated files.

In [ ]:
from pathlib import Path

PDF_ROOT = PROJ_ROOT / "res" / "PDF-20260721"
SERVICE_BASE_URL = os.getenv(
    "UNLIMITED_OCR_BASE_URL",
    _DEFAULT_BASE_URL,
)
OCR_API_KEY = os.getenv("UNLIMITED_OCR_API_KEY") or None
OCR_MODEL = os.getenv("UNLIMITED_OCR_MODEL") or None
OUTPUT_DIR_NAME = ".md_unlimited"
OCR_DPI = 300
REQUEST_TIMEOUT_S = 1200
MAX_CONCURRENCY = int(os.getenv("UNLIMITED_OCR_MAX_CONCURRENCY", "4"))
PAGE_CONCURRENCY = int(os.getenv("UNLIMITED_OCR_PAGE_CONCURRENCY", "2"))
MAX_TOKENS = 8192
OVERWRITE = False

In [ ]:
# | notest
# Confirm credentials and the served model name:
# await list_served_models(
#     base_url=SERVICE_BASE_URL,
#     api_key=OCR_API_KEY,
# )

# Run the recursive batch:
# results = await ocr_folder(
#     PDF_ROOT,
#     base_url=SERVICE_BASE_URL,
#     api_key=OCR_API_KEY,
#     model=OCR_MODEL,
#     output_dir_name=OUTPUT_DIR_NAME,
#     dpi=OCR_DPI,
#     overwrite=OVERWRITE,
#     request_timeout_s=REQUEST_TIMEOUT_S,
#     max_concurrency=MAX_CONCURRENCY,
#     page_concurrency=PAGE_CONCURRENCY,
#     max_tokens=MAX_TOKENS,
# )

## Tests

Tests use small generated documents and a fake OpenAI-compatible client. They
make no network calls and do not write to the repository.

In [ ]:
# | hide
from contextlib import redirect_stdout
from io import StringIO
from types import SimpleNamespace
from unittest.mock import patch

from fastcore.test import test_eq


class FakeAsyncOpenAIClient:
    def __init__(self, responses=(), delay_s=0.001):
        self.responses = list(responses)
        self.delay_s = delay_s
        self.calls = []
        self.active_calls = 0
        self.max_active_calls = 0
        self.closed = False
        self.chat = SimpleNamespace(
            completions=SimpleNamespace(create=self._create)
        )

    async def _create(self, **kwargs):
        self.calls.append(kwargs)
        if not self.responses:
            raise AssertionError("Unexpected chat completion call")
        response = self.responses.pop(0)
        delay_s = self.delay_s
        if isinstance(response, tuple):
            delay_s, response = response
        self.active_calls += 1
        self.max_active_calls = max(self.max_active_calls, self.active_calls)
        try:
            await asyncio.sleep(delay_s)
        finally:
            self.active_calls -= 1
        if isinstance(response, Exception):
            raise response
        if not isinstance(response, str):
            return response
        call_number = len(self.calls)
        return SimpleNamespace(
            id=f"response-{call_number}",
            choices=[
                SimpleNamespace(
                    message=SimpleNamespace(content=response),
                    finish_reason="stop",
                )
            ],
            usage=SimpleNamespace(
                prompt_tokens=100 + call_number,
                completion_tokens=10 + call_number,
                total_tokens=110 + 2 * call_number,
            ),
        )

    async def close(self):
        self.closed = True


def make_pdf(path: Path, labels=("page",), password: str | None = None):
    document = pymupdf.open()
    for label in labels:
        page = document.new_page(width=240, height=120)
        page.insert_text((24, 60), label)
    if password is None:
        document.save(path)
    else:
        document.save(
            path,
            encryption=pymupdf.PDF_ENCRYPT_AES_256,
            owner_pw="owner-password",
            user_pw=password,
        )
    document.close()


def make_image(path: Path, label: str = "image"):
    document = pymupdf.open()
    page = document.new_page(width=240, height=120)
    page.insert_text((24, 60), label)
    page.get_pixmap(alpha=False).save(path)
    document.close()

In [ ]:
# | hide
def test_configuration_and_grounding():
    test_eq(_resolve_base_url("172.27.74.16:7870"), "http://172.27.74.16:7870/v1")
    test_eq(
        _resolve_base_url("http://172.27.74.16:7870/v1/"),
        "http://172.27.74.16:7870/v1",
    )
    test_eq(_resolve_model(None), "baidu/Unlimited-OCR")
    test_eq(
        clean_grounding(
            "<|ref|># Heading<|/ref|><|det|>[[1,2,3,4]]<|/det|>\n\nBody"
        ),
        "# Heading\n\nBody",
    )
    test_eq(
        _vllm_extra_body(35, 128),
        {
            "skip_special_tokens": False,
            "vllm_xargs": {"ngram_size": 35, "window_size": 128},
        },
    )
    for invalid in ("", "document parsing.", " <imagex>bad"):
        try:
            _resolve_prompt(invalid)
        except ValueError:
            pass
        else:
            raise AssertionError(f"Expected invalid prompt: {invalid!r}")


test_configuration_and_grounding()

In [ ]:
# | hide
async def test_pdf_payload_order_and_sidecar():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        source = root / "two-pages.pdf"
        target = root / "two-pages.md"
        make_pdf(source, ("one", "two"))
        first = (
            "<|ref|># One<|/ref|>"
            "<|det|>[[0,0,999,100]]<|/det|>\n\nFirst body"
        )
        second = "<|ref|>## Two<|/ref|><|det|>[[0,0,999,100]]<|/det|>"
        client = FakeAsyncOpenAIClient(((0.03, first), (0.005, second)))
        result = await ocr_pdf(
            source,
            target,
            client=client,
            dpi=72,
            page_concurrency=2,
        )
        assert result.status == "processed", result.error
        test_eq(result.pages_total, 2)
        test_eq(result.pages_completed, 2)
        test_eq(
            target.read_text(encoding="utf-8"),
            "<!-- Page 1 -->\n\n# One\n\nFirst body\n\n"
            "<!-- Page 2 -->\n\n## Two\n",
        )
        assert client.max_active_calls == 2

        for call in client.calls:
            test_eq(call["model"], "baidu/Unlimited-OCR")
            test_eq(call["max_tokens"], 8192)
            test_eq(call["temperature"], 0.0)
            test_eq(
                call["extra_body"],
                {
                    "skip_special_tokens": False,
                    "vllm_xargs": {"ngram_size": 35, "window_size": 128},
                },
            )
            content = call["messages"][0]["content"]
            test_eq(content[0], {"type": "text", "text": "<image>document parsing."})
            image_url = content[1]["image_url"]["url"]
            assert image_url.startswith("data:image/png;base64,")
            assert base64.b64decode(image_url.split(",", 1)[1]).startswith(b"\x89PNG")

        metadata = json.loads(
            target.with_suffix(".unlimited.json").read_text(encoding="utf-8")
        )
        test_eq(metadata["provider"], "baidu/Unlimited-OCR")
        test_eq(metadata["service_base_url"], "http://172.27.74.16:7870/v1")
        test_eq(metadata["pages"][0]["raw_content"], first)
        test_eq(metadata["pages"][1]["content"], "## Two")
        assert "api_key" not in metadata


await test_pdf_payload_order_and_sidecar()

In [ ]:
# | hide
async def test_image_skip_failure_and_folder_concurrency():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        source = root / "图像.png"
        target = root / "结果.md"
        make_image(source)
        first = await ocr_image(
            source,
            target,
            client=FakeAsyncOpenAIClient(("recognized",)),
        )
        test_eq(first.status, "processed")
        original_markdown = target.read_bytes()
        original_metadata = target.with_suffix(".unlimited.json").read_bytes()

        skipped = await ocr_image(
            source,
            target,
            client=FakeAsyncOpenAIClient(),
        )
        test_eq(skipped.status, "skipped")

        failed = await ocr_image(
            source,
            target,
            client=FakeAsyncOpenAIClient((RuntimeError("service failure"),)),
            overwrite=True,
        )
        test_eq(failed.status, "failed")
        test_eq(target.read_bytes(), original_markdown)
        test_eq(target.with_suffix(".unlimited.json").read_bytes(), original_metadata)

    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        for name in ("a.png", "b.png", "c.png", "skipped.png"):
            make_image(root / name, name)
        output_root = root / ".md_unlimited"
        output_root.mkdir()
        (output_root / "skipped.md").write_text("existing", encoding="utf-8")
        client = FakeAsyncOpenAIClient(
            ((0.02, "A"), RuntimeError("B failed"), (0.02, "C"))
        )
        output = StringIO()
        with redirect_stdout(output):
            results = await ocr_folder(
                root,
                client=client,
                max_concurrency=2,
                page_concurrency=2,
                show_progress=False,
            )
        test_eq(
            [result.status for result in results],
            ["processed", "failed", "processed", "skipped"],
        )
        assert 1 < client.max_active_calls <= 2
        test_eq(len(client.calls), 3)
        assert (output_root / "a.md").is_file()
        assert not (output_root / "b.md").exists()
        assert (output_root / "c.unlimited.json").is_file()
        assert "1 skipped" in output.getvalue()
        assert not client.closed


await test_image_skip_failure_and_folder_concurrency()

In [ ]:
# | hide
async def test_invalid_inputs():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        corrupt = root / "corrupt.pdf"
        corrupt.write_bytes(b"not a PDF")
        corrupt_result = await ocr_pdf(
            corrupt,
            root / "corrupt.md",
            client=FakeAsyncOpenAIClient(),
        )
        test_eq(corrupt_result.status, "failed")

        encrypted = root / "encrypted.pdf"
        make_pdf(encrypted, password="secret")
        encrypted_result = await ocr_pdf(
            encrypted,
            root / "encrypted.md",
            client=FakeAsyncOpenAIClient(),
        )
        test_eq(encrypted_result.status, "failed")
        assert "password" in encrypted_result.error.casefold()

        image = root / "empty.png"
        make_image(image)
        empty_response = SimpleNamespace(
            choices=[
                SimpleNamespace(
                    message=SimpleNamespace(content="  "),
                    finish_reason="stop",
                )
            ]
        )
        empty_result = await ocr_image(
            image,
            root / "empty.md",
            client=FakeAsyncOpenAIClient((empty_response,)),
        )
        test_eq(empty_result.status, "failed")

        for kwargs, expected in (
            ({"dpi": 0}, "dpi"),
            ({"request_timeout_s": 0}, "request_timeout"),
            ({"max_concurrency": 0}, "MAX_CONCURRENCY"),
            ({"page_concurrency": 0}, "PAGE_CONCURRENCY"),
            ({"output_dir_name": "a/b"}, "output_dir_name"),
            ({"prompt": "document parsing."}, "<image>"),
            ({"model": " "}, "model"),
            ({"max_tokens": 0}, "max_tokens"),
        ):
            try:
                await ocr_folder(root, client=FakeAsyncOpenAIClient(), **kwargs)
            except Exception as error:
                assert expected.casefold() in str(error).casefold()
            else:
                raise AssertionError(f"Expected failure containing {expected!r}")


await test_invalid_inputs()